In [24]:
import datasets
from datasets import Dataset
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [79]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


In [85]:
model = SentenceTransformer("IEITYuan/Yuan-embedding-2.0-en", device="mps")
print(next(model.parameters()).device)

mps:0


In [38]:
embeddings = model.encode(["The wind blew strong and the storm was terrifying. The ship could sink at any moment",
              "It was so windy and rainy. The storm threatened us. We had to run under a tree to find cover.", 
              "I was riding my horse in the forest, when a knight started a fight, himmself on a horse.", 
                           "The two knights were fighting on the back of their horse, in a deadly tournament."], normalize_embeddings=True)

In [51]:
paragraphs = load_dataset("parquet", data_files = "../data/interim/paragraphs.parquet", split="train")

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

In [108]:
first_10 = paragraphs["paragraphs"][10000:12000]

In [109]:
%%timeit
embeddings = model.encode(first_10, normalize_embeddings=True)

1min 9s ± 1.49 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [110]:
24950946/2000

12475.473

In [111]:
9/60

0.15

In [112]:
12475.473*1.15/60

239.11323249999998

In [107]:
223/24

9.291666666666666

In [78]:
import torch

print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

True
True


# Processing the paragraphs

In [ ]:
uv pip install nltk

In [7]:
import nltk
from nltk.tokenize import sent_tokenize
#nltk.download()

In [3]:
paragraphs = load_dataset("parquet", data_files = "../data/interim/paragraphs.parquet", split="train")

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

In [14]:
def split_paragraph(paragraph: str, max_p_length: int) -> list[str]:
    """
    Splits a paragraph in as many needed to be smaller than the maximum length.
    :param paragraphs: content of the paragraph.
    :param max_p_length: maximum size of the paragraph.
    return at least one paragraph (if unchanged) or more (if needed to break)
    """
    all_subparagraphs = []
    sentences = sent_tokenize(paragraph)
    agg_sentences = []
    length_sentence = 0
    for sentence_number, sent in enumerate(sentences):
        words = sent.split()
        length_sent = len(words)
        length_sentence += length_sent
        if length_sentence > max_p_length and sentence_number > 0:
            all_subparagraphs.append(" ".join(agg_sentences))
            agg_sentences = [sent]
            length_sentence = length_sent
            continue
        
        agg_sentences.append(sent)
    
    all_subparagraphs.append(" ".join(agg_sentences))
    
    return all_subparagraphs
            
            

def break_paragraphs(paragraphs: str, max_p_length: int) -> str:
    """
    This function breaks paragraphs that are too big.
    :param paragraphs: content of the paragraph.
    :param max_p_length: maximum size of the paragraph.
    return at least one paragraph (if unchanged) or more (if needed to break)
    """
    splitted_paragraphs_data = {"paragraphs_broken":[], "n_words":[]}
    for p in paragraphs:
        all_words = p.split()
        length_p = len(all_words)
        if length_p < max_p_length:
            splitted_paragraphs_data["paragraphs_broken"].append(p)
            splitted_paragraphs_data["n_words"].append(len(all_words))
        else:
            number_of_cuts = length_p // max_p_length
            rest_of_cuts = length_p % max_p_length
            if rest_of_cuts > 0:
                number_of_cuts += 1
                
            print(max_p_length/number_of_cuts)
            splitted_paragraphs = split_paragraph(p, max_p_length/number_of_cuts)
            n_paragraphs = [len(p.split()) for p in splitted_paragraphs]
            splitted_paragraphs_data["paragraphs_broken"] += splitted_paragraphs
            splitted_paragraphs_data["n_words"] += n_paragraphs
            
    return splitted_paragraphs_data


def aggregate_paragraphs(hf_dataset: Dataset, threshold_min: int) -> Dataset:
    """
    Function that aggregates the paragraphs with sliding windows to get the context and a minimal amount
    of words for each aggregate.
    :param hf_dataset, the dataset to aggregate. Should have at least rows ["text_ids", "index", "paragraphs", "n_words"]
    :param threshold_min: integer, minimum number of words in the paragraph.
    """
    data = {col_num:[] for col_num in hf_dataset.features.keys()}
    current_length = 0
    current_text = ""
    #For each paragraph, if the total length paragraph is not up to threshold, add it.
    text_id = None
    paragraph_id = None
    chapter = None
    span_start = None
    span_stop = None
    for paragraph in hf_dataset:
        text_id_new = paragraph["text_ids"]
        paragraph_id_new = paragraph["paragraph_index"]
        chapter_new = paragraph["chapter"]
        if span_start is None:
            span_start = paragraph["spans"][0]
        if ((text_id == text_id_new or text_id is None) 
            and (paragraph_id_new == paragraph_id+1 or paragraph_id None)
            and chapter == chapter_new):
            current_length += paragraph["n_words"]
            current_text = "\n\n".join(current_text, paragraphs["paragraphs"])
            if current_length >=threshold_min:
                span_stop = span_start = paragraph["spans"][1]
                data["paragraphs"].append(current_text)
                data["n_words"].append(current_length)
                data["text_ids"].append(text_ids)
                data["chapters"].append(chapters)
                data["spans"].append((span_start, span_stop))
                span_start = None
                span_stop = None
                current_text = ""
                current_length = 0
                
        else:
            span_stop = span_start = paragraph["spans"][1]
            data["paragraphs"].append(current_text)
            data["n_words"].append(current_length)
            data["text_ids"].append(text_ids)
            data["chapters"].append(chapters)
            data["spans"].append((span_start, span_stop))
            span_start = None
            span_stop = None
            current_text = ""
            current_length = 0
                  

In [ ]:
['paragraphs', 'text_ids', 'spans', 'chapters', 'index']

In [ ]:
dict_keys(['paragraphs', 'text_ids', 
           'spans', 'chapters', 
           'index', 'n_words', 
           'paragraph_index', 'splitted_paragraphs'])

In [15]:
split_paragraph("This is a very long sentence, long long long long. It should be. It is small. Small sentence.", 16)

['This is a very long sentence, long long long long. It should be. It is small.',
 'Small sentence.']

In [16]:
l = ["This is a very long sentence, long long long long. It should be. It is small. Small sentence."]

In [19]:
93//2

46

In [20]:
93/46

2.0217391304347827

In [23]:
break_paragraphs(l, 2)

0.2222222222222222


{'paragraphs_broken': ['This is a very long sentence, long long long long.',
  'It should be.',
  'It is small.',
  'Small sentence.'],
 'n_words': [10, 3, 3, 2]}

In [12]:
len("This is a very long sentence, long long long long. It should be. It is small. Small sentence.")//16

5

In [13]:
len("This is a very long sentence, long long long long. It should be. It is small. Small sentence.")%16

13

In [10]:
len('This is a very long sentence, long long long long. It should be. It is small.')

77

In [7]:
break_paragraphs(["This is a very long sentence, long long long long. It should be. It is small. Small sentence."], 17)

{'paragraphs': ['This is a very long sentence, long long long long. It should be. It is small.',
  'Small sentence.'],
 'n_words': [16, 2]}

In [8]:
arr = ["ajo joi", "jioj k"]
result = np.char.split(arr, sep=" ")

In [9]:
result

array([list(['ajo', 'joi']), list(['jioj', 'k'])], dtype=object)

In [10]:
len("This is a very long sentence, long long long long. It should be. It is small.".split( ))

16

In [11]:
len("This is a very long sentence, long long long long. It should be. It is small. Small sentence.".split( ))

18

In [2]:
1050//1000

1

In [3]:
3050//1000

3

In [4]:
3050%1000

50

In [16]:
p = paragraphs.map(break_paragraphs, num_proc=4, batched=True, batch_size=8, 
               remove_columns=["paragraphs", "text_ids", 'spans', 'chapters'], fn_kwargs={"max_p_length":1000})

Exception ignored in: <function Dataset.__del__ at 0x111a45f80>
Traceback (most recent call last):
  File "/Users/gabdu45/ProjectAlbalat/.venv/lib/python3.11/site-packages/datasets/arrow_dataset.py", line 1780, in __del__
    if hasattr(self, "_indices"):
       ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 
Exception ignored in: <function Dataset.__del__ at 0x111a45f80>
Traceback (most recent call last):
  File "/Users/gabdu45/ProjectAlbalat/.venv/lib/python3.11/site-packages/datasets/arrow_dataset.py", line 1780, in __del__
    if hasattr(self, "_indices"):
       ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 


In [94]:
help(datasets.Dataset.map)

Help on function map in module datasets.arrow_dataset:

map(self, function: Optional[Callable] = None, with_indices: bool = False, with_rank: bool = False, input_columns: Union[str, list[str], NoneType] = None, batched: bool = False, batch_size: Optional[int] = 1000, drop_last_batch: bool = False, remove_columns: Union[str, list[str], NoneType] = None, keep_in_memory: bool = False, load_from_cache_file: Optional[bool] = None, cache_file_name: Optional[str] = None, writer_batch_size: Optional[int] = 1000, features: Optional[datasets.features.features.Features] = None, disable_nullable: bool = False, fn_kwargs: Optional[dict] = None, num_proc: Optional[int] = None, suffix_template: str = '_{rank:05d}_of_{num_proc:05d}', new_fingerprint: Optional[str] = None, desc: Optional[str] = None, try_original_type: Optional[bool] = True, on_mixed_types: Optional[Literal['use_json']] = 'use_json') -> 'Dataset'
    Apply a function to all the examples in the table (individually or in batches) and upd

In [104]:
paragraphs

Dataset({
    features: ['paragraphs', 'text_ids', 'spans', 'chapters'],
    num_rows: 24950946
})

In [16]:
from datasets.utils.logging import is_progress_bar_enabled

print(is_progress_bar_enabled())

True


In [20]:
p

NameError: name 'p' is not defined

In [23]:
from tqdm.auto import tqdm
import time

for _ in tqdm(range(100)):
    time.sleep(0.02)

  0%|          | 0/100 [00:00<?, ?it/s]

In [24]:
print("Hello")

Hello


In [25]:
from tqdm.notebook import tqdm
import time

for _ in tqdm(range(100)):
    time.sleep(0.05)

  0%|          | 0/100 [00:00<?, ?it/s]

In [26]:
import notebook
import jupyterlab
import tqdm
import ipywidgets

print("tqdm:", tqdm.__version__)

try:
    print("notebook:", notebook.__version__)
except:
    print("notebook: not installed")

try:
    print("jupyterlab:", jupyterlab.__version__)
except:
    print("jupyterlab: not installed")

try:
    print("ipywidgets:", ipywidgets.__version__)
except:
    print("ipywidgets: not installed")

tqdm: 4.68.3
notebook: 7.6.0
jupyterlab: 4.6.0
ipywidgets: 8.1.8


In [27]:
from IPython.display import display
import ipywidgets as widgets

display(widgets.IntProgress(value=50, min=0, max=100))

IntProgress(value=50)

In [29]:
from tqdm.notebook import tqdm
import time

bar = tqdm(total=100)
for i in range(100):
    time.sleep(0.05)
    bar.update(1)
bar.close()

  0%|          | 0/100 [00:00<?, ?it/s]

In [25]:
"a" == None

False

In [26]:
dict_data = {"paragraphs":["This is a test paragraphs.",
                           "and another one",
                           "yet another one",
                           "another small one"
                           "a very very very very very very very very very very long paragraph."
                           ],
             "chapters":["I, I, II, II, II"],
             "n_words":[5, 3, 3, 3, 13],
             "text_ids":[0, 0, 1, 1, 1]}

hf_dataset = Dataset.from_dict(dict_data)

ArrowInvalid: Column 1 named chapters expected length 4 but got length 1

In [28]:
dict_data["chapters"]

['I, I, II, II, II']